# Dokumentasi Catatan & Tanya-Jawab Kriptografi
## Persiapan Diskusi Kelompok 1 — MA4151 Kriptografi
---
Dokumen ini memuat catatan penjelasan teoritis, penurunan matematis, serta kode verifikasi interaktif untuk pertanyaan-pertanyaan seputar **Sandi Klasik**, **Kriptoanalisis**, dan **Teori Shannon**.

## 📌 Topik 1: Perbedaan Kriptoanalisis Affine vs. Kriptoanalisis Vigenère

### 1.1 Tabel Perbandingan Karakteristik

| Parameter | Kriptoanalisis Sandi Affine | Kriptoanalisis Sandi Vigenère |
| :--- | :--- | :--- |
| **Kategori Sandi** | **Monoalfabetik** (1 huruf plaintext $\to$ 1 huruf ciphertext yang sama). | **Polialfabetik Periodik** (1 huruf plaintext dipetakan berbeda tergantung posisinya). |
| **Formulasi Enkripsi** | $c \equiv a \cdot p + b \pmod{26}$, dengan $\gcd(a, 26) = 1$. | $c_i \equiv p_i + k_{i \bmod m} \pmod{26}$, dengan kunci $\mathbf{k} = (k_0, \dots, k_{m-1})$. |
| **Ruang Kunci ($|\mathcal{K}|$)** | $|\mathcal{K}| = \varphi(26) \times 26 = 12 \times 26 = 312$. | $|\mathcal{K}| = 26^m$ (tumbuh eksponensial terhadap panjang kunci $m$). |
| **Metode Serangan KPA** | Cukup **2 pasang huruf** $(p_1, c_1)$ dan $(p_2, c_2)$ untuk membentuk SPL kongruensi 2 variabel modulo 26. | Butuh $m$ pasang huruf pada posisi indeks yang kongruen modulo $m$. |
| **Metode Serangan COA** | Memetakan 2 huruf tersering ciphertext ke 2 huruf tersering bahasa asli (misal $E, T$ atau $A, N$). | **Tidak bisa langsung analisis frekuensi**. Wajib cari panjang periode $m$ dulu (Kasiski/Friedman), lalu dekomposisi per kolom. |
| **Indeks Koinsidensi ($I_c$)** | Tinggi ($I_c \approx 0{,}065$ untuk Inggris, $\approx 0{,}074$ untuk Indonesia). | Rendah ($I_c \approx 0{,}038 - 0{,}045$) mendekati teks acak murni seragam. |

### 1.2 Prosedur Matematis Kriptoanalisis Sandi Affine (KPA)

Jika diketahui 2 pasangan karakter $p_1 \to c_1$ dan $p_2 \to c_2$:
$$
\begin{cases}
a \cdot p_1 + b \equiv c_1 \pmod{26} \\[4pt]
a \cdot p_2 + b \equiv c_2 \pmod{26}
\end{cases}
$$

1. **Eliminasi $b$:**
   $$a(p_1 - p_2) \equiv (c_1 - c_2) \pmod{26}$$
2. **Cari $a$:** Definisikan $\Delta p = (p_1 - p_2) \bmod 26$ dan $\Delta c = (c_1 - c_2) \bmod 26$. Jika $\gcd(\Delta p, 26) = 1$, maka:
   $$a \equiv \Delta c \cdot (\Delta p)^{-1} \pmod{26}$$
3. **Cari $b$:**
   $$b \equiv c_1 - a \cdot p_1 \pmod{26}$$
4. **Verifikasi:** Pastikan $\gcd(a, 26) = 1$ agar aturan dekripsi valid.

In [ ]:
# Contoh Komputasi Python: Pemecahan KPA Sandi Affine
from math import gcd

def mod_inverse(a, m=26):
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    return None

def solve_affine_kpa(p1_char, c1_char, p2_char, c2_char):
    p1, c1 = ord(p1_char.upper()) - ord('A'), ord(c1_char.upper()) - ord('A')
    p2, c2 = ord(p2_char.upper()) - ord('A'), ord(c2_char.upper()) - ord('A')
    
    delta_p = (p1 - p2) % 26
    delta_c = (c1 - c2) % 26
    
    if gcd(delta_p, 26) != 1:
        return f"Gagal: gcd({delta_p}, 26) = {gcd(delta_p, 26)} != 1 (delta_p tidak punya invers di mod 26)"
    
    inv_delta_p = mod_inverse(delta_p, 26)
    a = (delta_c * inv_delta_p) % 26
    b = (c1 - a * p1) % 26
    
    return {"a": a, "b": b, "gcd(a, 26)": gcd(a, 26)}

# Uji coba dengan Soal 12: 'IF' -> 'PQ' (I=8, F=5, P=15, Q=16)
hasil = solve_affine_kpa('I', 'P', 'F', 'Q')
print("Hasil KPA Affine 'IF' -> 'PQ':", hasil)

---
## 📌 Topik 2: Panduan Lengkap Kriptoanalisis Sandi Klasik & Urutan Metodenya

### 2.1 Model Penyerangan Formal
1. **Ciphertext-Only Attack (COA):** Hanya teks sandi $y$.
2. **Known-Plaintext Attack (KPA):** Diketahui pasangan $(x_i, y_i)$.
3. **Chosen-Plaintext Attack (CPA):** Penyerang dapat mengenkripsi sembarang $x$ pilihan.
4. **Chosen-Ciphertext Attack (CCA):** Penyerang dapat mendekripsi sembarang $y$ pilihan.

### 2.2 Tabel Pemetaan Sandi vs. Metode Kriptoanalisis

| Sandi | Jenis | Serangan | Data Minimum | Metode Utama |
| :--- | :--- | :--- | :--- | :--- |
| **Geser (*Shift*)** | Monoalfabetik | COA / KPA | 1 pasang / $N \ge 30$ | Brute-force 26 kunci atau pencocokan frekuensi unigram tertinggi. |
| **Affine** | Monoalfabetik | KPA / COA | 2 pasang / 2 huruf puncak | Sistem kongruensi linear 2 persamaan modulo 26. |
| **Hill ($m \times m$)** | Poligram Aljabar | KPA | $m$ pasang blok vektor | Aljabar matriks modular: $K \equiv Y X^{-1} \pmod{26}$. |
| **Transposisi** | Permutasi | COA | $N = n \cdot m$ | Faktorisasi panjang teks, analisis nilai $I_c \approx I_{\text{mono}}$, dan rekonstruksi urutan kolom via *bigram score*. |
| **Vigenère** | Polialfabetik | COA | $N \ge 100$ | Alur 5 Tahap: Uji $I_c \to$ Kasiski/Friedman (cari $m$) $\to$ Partisi Kolom $\to$ Analisis Korelasi Kolom $\to$ Dekripsi Penuh. |

### 2.3 Alur 5 Tahap Kriptoanalisis Sandi Vigenère

$$\boxed{\text{Tahap 0: Diagnostik } I_c} \longrightarrow \boxed{\text{Tahap 1: Cari Panjang Kunci } m} \longrightarrow \boxed{\text{Tahap 2: Partisi Kolom}} \longrightarrow \boxed{\text{Tahap 3: Pecahkan Tiap Kolom}} \longrightarrow \boxed{\text{Tahap 4: Dekripsi Penuh}}$$

#### **Tahap 0: Uji Diagnostik Indeks Koinsidensi ($I_c$)**
$$I_c(\mathbf{y}) = \frac{\sum_{i=0}^{25} f_i(f_i - 1)}{N(N - 1)}$$
- Jika $I_c \approx 0{,}065$ (Inggris) atau $\approx 0{,}074$ (Indonesia) $\implies$ Cipher monoalfabetik / transposisi.
- Jika $I_c \approx 0{,}038 - 0{,}045$ $\implies$ Cipher polialfabetik (Vigenère), lanjut ke Tahap 1.

#### **Tahap 1: Menentukan Panjang Periode Kunci ($m$)**
- **Uji Kasiski:** Cari $n$-gram berulang ($n \ge 3$), hitung selisih jarak kemunculan $D_{ij} = p_j - p_i$, lalu faktorisasi:
  $$m \mid \gcd(D_1, D_2, \dots, D_r)$$
- **Formula Babbage-Friedman:**
  $$m \approx \frac{0{,}027 N}{(N - 1)I_c - 0{,}038 N + 0{,}065}$$
- **Rata-Rata Indeks Koinsidensi Kolom $\bar{I}_c(m)$:** Uji calon $m = 1, 2, 3, \dots$ hingga rata-rata $I_c$ per kolom melonjak mendekati $\approx 0{,}065$.

#### **Tahap 2: Dekomposisi Menjadi $m$ Kolom Monoalfabetik**
Ekstrak $m$ sub-vektor kolom $\mathbf{y}^{(0)}, \mathbf{y}^{(1)}, \dots, \mathbf{y}^{(m-1)}$:
$$\mathbf{y}^{(j)} = (y_j, \, y_{j+m}, \, y_{j+2m}, \, \dots) \quad \text{untuk } j \in \{0, 1, \dots, m-1\}$$
Tiap kolom $\mathbf{y}^{(j)}$ adalah Sandi Geser dengan satu huruf kunci $k_j$.

#### **Tahap 3: Pemecahan Huruf Kunci Tiap Kolom ($k_j$)**
- **Teknik 1 (Korelasi Bahasa):** $M_j(k) = \sum_{i=0}^{25} p_i \cdot \frac{f_{i+k}^{(j)}}{N_j}$ dan pilih $k_j = \arg\max_k M_j(k)$.
- **Teknik 2 (Mutual Index of Coincidence - $MIC$):** $MIC(\mathbf{y}^{(a)}, \mathbf{y}^{(b)}, g) = \frac{\sum f_{a,i} f_{b, (i+g)\bmod 26}}{N_a N_b}$ untuk mencari selisih relatif $k_b - k_a \equiv g_{\max} \pmod{26}$.
- **Teknik 3 (Frekuensi Puncak Cepat):** $k_j \equiv y_{\max}^{(j)} - x_{\max} \pmod{26}$.

#### **Tahap 4: Rekonstruksi Kunci & Dekripsi Penuh**
Gabungkan $\mathbf{k} = (k_0, k_1, \dots, k_{m-1})$, lalu dekripsi:
$$x_i \equiv y_i - k_{i \bmod m} \pmod{26}$$

In [ ]:
# Demonstrasi Python: Perhitungan Indeks Koinsidensi (Ic) dan Estimasi Friedman
from collections import Counter

def hitung_ic(teks):
    teks = [c.upper() for c in teks if c.isalpha()]
    N = len(teks)
    if N <= 1:
        return 0
    freq = Counter(teks)
    sum_fi = sum(f * (f - 1) for f in freq.values())
    return sum_fi / (N * (N - 1))

def estimasi_friedman(N, ic):
    pembilang = 0.027 * N
    penyebut = (N - 1) * ic - 0.038 * N + 0.065
    if penyebut <= 0:
        return None
    return pembilang / penyebut

# Contoh: Ciphertext N=500, Ic=0.0450 (Soal 11)
m_est = estimasi_friedman(500, 0.0450)
print(f"Estimasi panjang kunci Friedman (N=500, Ic=0.0450): m ≈ {m_est:.4f} ≈ {round(m_est)}")

---
## 📌 Topik 3: Pengujian Keamanan Sempurna (Perfect Secrecy) — Teori Shannon

### 3.1 Apakah Benar Ada Dua Cara Pengujian (PMF vs. Entropi)?

**BENAR SEKALI!** Kedua pendekatan tersebut saling ekuivalen secara matematis, ditambah satu cara ketiga (jalan pintas struktural Teorema Karakterisasi Shannon):

```
                 ┌─────────────────────────────────────────────────────────┐
                 │           UJI KEAMANAN SEMPURNA (SHANNON)               │
                 └────────────────────────────┬────────────────────────────┘
                                              │
     ┌────────────────────────────────────────┼────────────────────────────────────────┐
     ▼                                        ▼                                        ▼
【 CARA 1: PMF 】                     【 CARA 2: ENTROPI 】                   【 CARA 3: STRUKTUR KUNCI 】
(Distribusi Bersyarat)                 (Teori Informasi)                   (Teorema Karakterisasi Shannon)
• p(x | y) = p(x)                      • H(P | C) = H(P)                   • Syarat: |K| = |C| = |P|
• p(y | x) = p(y)                      • I(P; C) = 0                       • p(k) = 1/|K| (seragam)
• P dan C saling bebas                 • H(C | P) = H(C)                   • Tepat 1 kunci e_k(x) = y
```

### 3.2 Rincian Tiga Cara Pengujian

#### **1. Pendekatan Distribusi Peluang / PMF (Definisi Asli Shannon)**
Mengetahui ciphertext $y$ sama sekali tidak mengubah keyakinan/peluang terhadap plaintext $x$:
$$\forall x \in \mathcal{P}, \forall y \in \mathcal{C} \quad (p(y) > 0): \quad p(x \mid y) = p(x)$$
Secara ekuivalen:
$$\forall x \in \mathcal{P}, \forall y \in \mathcal{C}: \quad p(y \mid x) = p(y) \iff p(x, y) = p(x) \cdot p(y)$$
> **Tips Praktis:** Sangat efektif untuk **membuktikan sistem TIDAK aman sempurna** (cukup temukan satu *counterexample* $p(x_0 \mid y_0) \neq p(x_0)$).

#### **2. Pendekatan Entropi / Teori Informasi (*Message Equivocation*)**
Ketidakpastian mengenai plaintext setelah menyadap ciphertext tidak berkurang sama sekali:
$$H(P \mid C) = H(P)$$
Secara ekuivalen (*Mutual Information* bernilai nol):
$$I(P; C) = H(P) - H(P \mid C) = 0 \iff H(P, C) = H(P) + H(C)$$
> **Tips Praktis:** Gunakan rumus sakti *Key Equivocation*:
> $$H(K \mid C) = H(K) + H(P) - H(C)$$
> Pada sistem di mana untuk setiap $(x, y)$ terdapat tepat 1 kunci $k$, berlaku $H(P \mid C) = H(K \mid C)$.

#### **3. Pendekatan Karakterisasi Shannon (Pintasan Struktural saat $|\mathcal{K}| = |\mathcal{C}| = |\mathcal{P}|$ )**
Jika ukuran ruang kunci sama dengan ruang pesan dan ruang ciphertext, sistem aman sempurna jika dan hanya jika:
1. Pemilihan kunci seragam: $p(k) = \frac{1}{|\mathcal{K}|}$ untuk setiap $k \in \mathcal{K}$.
2. Untuk setiap $x \in \mathcal{P}$ dan $y \in \mathcal{C}$, terdapat **tepat satu** $k \in \mathcal{K}$ dengan $e_k(x) = y$.

In [ ]:
# Demonstrasi Python: Perhitungan Entropi Shannon dan Equivocation
import math
import numpy as np

def entropy(probs):
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Simulasi Soal 15 & 16:
# P = {a, b, c}, K = {K1, K2, K3}, C = {1, 2, 3, 4}
p_P = {'a': 1/2, 'b': 1/3, 'c': 1/6}
p_K = {'K1': 1/3, 'K2': 1/3, 'K3': 1/3}

# Enkripsi matriks: e(K, P)
enc = {
    ('K1', 'a'): 1, ('K1', 'b'): 2, ('K1', 'c'): 3,
    ('K2', 'a'): 2, ('K2', 'b'): 3, ('K2', 'c'): 4,
    ('K3', 'a'): 3, ('K3', 'b'): 4, ('K3', 'c'): 1,
}

# Hitung marginal p(C)
p_C = {1: 0, 2: 0, 3: 0, 4: 0}
for (k, x), y in enc.items():
    p_C[y] += p_P[x] * p_K[k]

H_P = entropy(p_P.values())
H_K = entropy(p_K.values())
H_C = entropy(p_C.values())
H_K_given_C = H_K + H_P - H_C  # Rumus sakti Key Equivocation

print("Distribusi Marginal p(C):", {k: f"{v:.4f} ({v})" for k, v in p_C.items()})
print(f"H(P)     = {H_P:.4f} bit")
print(f"H(K)     = {H_K:.4f} bit")
print(f"H(C)     = {H_C:.4f} bit")
print(f"H(K | C) = {H_K_given_C:.4f} bit")
print(f"Apakah H(P | C) == H(P)? {abs(H_K_given_C - H_P) < 1e-9} (Sistem {'AMAN SEMPURNA' if abs(H_K_given_C - H_P) < 1e-9 else 'TIDAK AMAN SEMPURNA'})")

---
## 📌 Topik 4: Sandi yang Memakai Persamaan Rekursif (Relasi Rekurensi)

Dalam kriptografi, terdapat beberapa sandi yang memakai **persamaan rekursif (relasi rekurensi)**:

### 4.1 Sandi Aliran Rekursif Linier (*Linear Recurrence / LFSR Stream Cipher*)
Ini adalah kelompok sandi yang paling umum dan formal menggunakan **relasi rekurensi linier** (persamaan diferensi berderajat $m$) untuk membangkitkan aliran kunci (*keystream* $z_1, z_2, z_3, \dots$):

$$
z_{i+m} \equiv \sum_{j=0}^{m-1} c_j z_{i+j} = c_{m-1} z_{i+m-1} + c_{m-2} z_{i+m-2} + \dots + c_0 z_i \pmod{26} \quad (\text{atau } \pmod 2)
$$

- **Struktur Kunci:**
  1. Nilai awal (*seed / initial state*): $\mathbf{z}_0 = (z_0, z_1, \dots, z_{m-1}) \in \mathbb{Z}_{26}^m$.
  2. Koefisien rekursi / koneksi: $\mathbf{c} = (c_0, c_1, \dots, c_{m-1}) \in \mathbb{Z}_{26}^m$.
- **Proses Enkripsi & Dekripsi:**
  $$c_i \equiv p_i + z_i \pmod{26}, \qquad p_i \equiv c_i - z_i \pmod{26}$$
- **Implementasi Perangkat Keras:** Sandi ini diwujudkan dalam sirkuit perangkat keras sebagai **LFSR (*Linear Feedback Shift Register*)**.

### 4.2 Sandi Kunci Otomatis (*Autokey Cipher*)
Sandi klasik di mana pembangkitan kunci berikutnya bergantung secara rekursif pada **karakter pesan sebelumnya atau teks sandi sebelumnya**:
1. **Plaintext Autokey:**
   $$k_i = \begin{cases} K_i & \text{jika } 0 \le i < m \text{ (kata kunci primer)} \\[4pt] p_{i-m} & \text{jika } i \ge m \text{ (rekursif dari plaintext)} \end{cases}$$
   $$\implies c_i \equiv p_i + p_{i-m} \pmod{26} \quad (i \ge m)$$
2. **Ciphertext Autokey (*Self-Synchronizing Stream Cipher*):**
   $$k_i = c_{i-1} \implies c_i \equiv p_i + c_{i-1} \pmod{26}$$

### 4.3 Kriptoanalisis Sandi Rekursif Linier via KPA (Sistem Matriks Hankel)
Jika penyerang mengetahui $2m$ karakter plaintext dan ciphertext, penyerang dapat menghitung $2m$ karakter keystream $z_i \equiv c_i - p_i \pmod{26}$.
Koefisien rekursi $(c_0, \dots, c_{m-1})$ dapat dicari dengan menyelesaikan sistem matriks linear Hankel:
$$
\begin{pmatrix}
z_0 & z_1 & \dots & z_{m-1} \\[3pt]
z_1 & z_2 & \dots & z_m \\[3pt]
\vdots & \vdots & \ddots & \vdots \\[3pt]
z_{m-1} & z_m & \dots & z_{2m-2}
\end{pmatrix}
\begin{pmatrix}
c_0 \\[3pt] c_1 \\[3pt] \vdots \\[3pt] c_{m-1}
\end{pmatrix}
\equiv
\begin{pmatrix}
z_m \\[3pt] z_{m+1} \\[3pt] \vdots \\[3pt] z_{2m-1}
\end{pmatrix}
\pmod{26}
$$

In [ ]:
# Demonstrasi Python: Sandi Aliran Rekursif Linier (Linear Recurrence Keystream)
def generate_linear_keystream(seed, coeffs, length, mod=26):
    m = len(seed)
    z = list(seed)
    for i in range(length - m):
        # z_{i+m} = c_0*z_i + c_1*z_{i+1} + ... + c_{m-1}*z_{i+m-1} mod 26
        next_val = sum(coeffs[j] * z[i + j] for j in range(m)) % mod
        z.append(next_val)
    return z

# Contoh: Orde m=2, Relasi Fibonacci: z_{i+2} = (z_{i+1} + z_i) mod 26
seed = [1, 3]       # z_0 = 1, z_1 = 3
coeffs = [1, 1]     # c_0 = 1, c_1 = 1
pesan = "KRIPTOGRAFI"

keystream = generate_linear_keystream(seed, coeffs, len(pesan), mod=26)
ciphertext = "".join(chr((ord(p) - ord('A') + k) % 26 + ord('A')) for p, k in zip(pesan, keystream))
dekripsi   = "".join(chr((ord(c) - ord('A') - k) % 26 + ord('A')) for c, k in zip(ciphertext, keystream))

print("Pesan Asli :", pesan)
print("Keystream  :", keystream)
print("Ciphertext :", ciphertext)
print("Hasil Dek  :", dekripsi)

---
## 📌 Topik 5: Klarifikasi Indeks Koinsidensi: Teks Utuh vs. Sub-Kolom Partisi

### 5.1 Pertanyaan Kunci
> *"Jika $I_c \approx 0{,}038 - 0{,}045$ menandakan polialfabetik, mengapa pada Soal 14b Dokumen 3 justru nilai $I_c \approx 0{,}0675$ (monoalfabetik) yang dinyatakan BENAR?"*

### 5.2 Perbedaan Konteks Pengujian: Teks Utuh vs. Sub-Kolom Partisi

| Kondisi Pengujian | Objek yang Dihitung | Nilai $I_c$ yang Muncul | Kesimpulan / Status |
| :--- | :--- | :--- | :--- |
| **Tahap 0: Diagnosis Teks Utuh** | Teks sandi utuh (belum dibagi) | $I_c \approx 0{,}038 - 0{,}045$ | Teks utuh terbukti **Polialfabetik (Vigenère)** $\implies$ Lanjut cari $m$. |
| **Tahap 1: Desimasi Sub-Kolom ($m=3$ salah)** | Rata-rata per kolom $\overline{IC}_3$ | $\overline{IC}_3 \approx 0{,}0443$ | **SALAH:** Karena tebakan $m=3$ keliru, huruf dalam 1 kolom masih tercampur banyak kunci (masih polialfabetik). |
| **Tahap 1: Desimasi Sub-Kolom ($m=5$ salah)** | Rata-rata per kolom $\overline{IC}_5$ | $\overline{IC}_5 \approx 0{,}0399$ | **SALAH:** Masih tercampur acak. |
| **Tahap 1: Desimasi Sub-Kolom ($m=4$ BENAR)** | Rata-rata per kolom $\overline{IC}_4$ | $\overline{IC}_4 \approx 0{,}0675 \approx I_{\text{mono}}$ | **BENAR ($m=4$):** Setiap kolom sekarang murni hanya dienkripsi oleh 1 huruf kunci (berubah menjadi **Sandi Geser Monoalfabetik**). |

### 5.3 Kesimpulan
- Pada **Teks Utuh**, kita mencari nilai **rendah** ($I_c \approx 0{,}038 - 0{,}045$) untuk memastikan teks tersebut memang **Vigenère**.
- Pada **Sub-Kolom Partisi**, kita mencari nilai **tinggi** ($I_c \approx 0{,}065 - 0{,}068$) karena panjang periode $m$ yang tepat berhasil **mengurai** Vigenère menjadi $m$ buah sandi monoalfabetik murni.

---
## 📌 Topik 6: Mutual Index of Coincidence (MIC)

### 6.1 Definisi Matematis
Jika $I_c$ standar mengukur koinsidensi internal pada satu teks/kolom, maka **Mutual Index of Coincidence ($MIC$)** mengukur derajat kesamaan antara **dua sub-kolom berbeda** $\mathbf{y}^{(a)}$ dan $\mathbf{y}^{(b)}$ dengan pergeseran relatif $g \in \mathbb{Z}_{26}$:

$$
MIC(\mathbf{y}^{(a)}, \mathbf{y}^{(b)}, g) = \frac{\sum_{i=0}^{25} f_{a, i} \cdot f_{b, (i+g) \bmod 26}}{N_a \cdot N_b}
$$

### 6.2 Peran Krusial $MIC$ dalam Kriptoanalisis Vigenère
1. **Mencari Pergeseran Relatif Antar Huruf Kunci:**
   Nilai $g_{\max} = \arg\max_g MIC(g)$ yang menghasilkan $MIC \approx 0{,}065$ memberikan hubungan:
   $$k_b - k_a \equiv g_{\max} \pmod{26} \iff k_b \equiv k_a + g_{\max} \pmod{26}$$
2. **Reduksi Ruang Kunci:**
   Dengan menghitung $MIC$ untuk pasangan $(Y_0, Y_1), (Y_0, Y_2), \dots, (Y_0, Y_{m-1})$, seluruh huruf kunci $k_1, k_2, \dots, k_{m-1}$ dapat dinyatakan dalam fungsi satu variabel $k_0$. Ruang pencarian kunci tereduksi dari $26^m$ menjadi **hanya 26 tebakan saja** (setara memecahkan Sandi Geser biasa).

In [ ]:
# Demonstrasi Python: Perhitungan Mutual Index of Coincidence (MIC)
def hitung_mic(subkolom_a, subkolom_b, g):
    from collections import Counter
    fa = Counter(ord(c.upper()) - ord('A') for c in subkolom_a if c.isalpha())
    fb = Counter(ord(c.upper()) - ord('A') for c in subkolom_b if c.isalpha())
    Na = sum(fa.values())
    Nb = sum(fb.values())
    
    pembilang = sum(fa[i] * fb[(i + g) % 26] for i in range(26))
    return pembilang / (Na * Nb)

# Simulasi Soal 15 Dokumen 3: N1 = 50, N2 = 50
# g=9 memiliki pembilang 172
mic_optimal = 172 / (50 * 50)
print(f"MIC pada pergeseran g=9: {mic_optimal:.4f} (Mendekati 0.065, sehingga k2 - k1 ≡ 9 mod 26)")

---
## 📌 Ruang Catatan Tambahan (Untuk Pertanyaan Selanjutnya)
*(Bagian ini dapat terus ditambahkan pembahasan soal baru, penurunan rumus, atau eksperimen kode kriptografi lainnya)*